In [13]:
import requests
import pandas as pd

In [14]:
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

In [15]:
session = requests.Session()

headers = {
    "User-Agent": "Mozilla/5.0",
    "Accept": "application/json",
    "Referer": "https://srienlinea.sri.gob.ec/saiku-ui/",
}

In [16]:
# obtener JSESSIONID
session.get(
    "https://srienlinea.sri.gob.ec/saiku/rest/saiku/session",
    headers=headers,
    verify=False
)

C:\Users\dfreirev\AppData\Roaming\Python\Python311\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'srienlinea.sri.gob.ec'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


<Response [200]>

In [17]:
def discover_dimensions():
    payload = {
        "datasource": "declaracion104",
        "cube": "[D104]",
        "type": "dimensions"
    }

    r = requests.post(
        f"{BASE_URL}/discover",
        headers=HEADERS,
        json=payload,
        timeout=30
    )
    r.raise_for_status()
    return r.json()

In [18]:
result_url = "https://srienlinea.sri.gob.ec/saiku/rest/saiku/anonymousUser/datasources"

r = session.get(result_url, headers=headers, verify=False)

print(r.status_code)
print(r.json())

200
[{'name': 'declaracion101', 'type': 'OLAP', 'properties': {'driver': 'mondrian.olap4j.MondrianOlap4jDriver', 'name': 'declaracion101', 'location': 'jdbc:mondrian:Jdbc=jdbc:oracle:thin:/@INTER_BOLETIN_DIGITAL;JdbcDrivers=oracle.jdbc.driver.OracleDriver;Catalog=res:declaracion/Declaraciones101.mondrian.xml;', 'type': 'OLAP'}}, {'name': 'recaudacion', 'type': 'OLAP', 'properties': {'driver': 'mondrian.olap4j.MondrianOlap4jDriver', 'name': 'recaudacion', 'location': 'jdbc:mondrian:Jdbc=jdbc:oracle:thin:/@INTERSRI_BI_DWH;JdbcDrivers=oracle.jdbc.driver.OracleDriver;Catalog=res:recaudacion/Recaudacion_Boletin_Estrella_Investigador.xml;', 'type': 'OLAP'}}, {'name': 'declaracion104', 'type': 'OLAP', 'properties': {'driver': 'mondrian.olap4j.MondrianOlap4jDriver', 'name': 'declaracion104', 'location': 'jdbc:mondrian:Jdbc=jdbc:oracle:thin:/@INTER_BOLETIN_DIGITAL;JdbcDrivers=oracle.jdbc.driver.OracleDriver;Catalog=res:declaracion/Declaraciones104.mondrian.xml;', 'type': 'OLAP'}}, {'name': 'Con

C:\Users\dfreirev\AppData\Roaming\Python\Python311\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'srienlinea.sri.gob.ec'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


In [19]:
# Suponiendo que ya tienes el JSON en r.json()
data = r.json()

# Aplanamos la estructura
rows = []
for ds in data:
    row = {
        "nombre": ds.get("name"),
        "tipo": ds.get("type"),
        "driver": ds["properties"].get("driver"),
        "jdbc_location": ds["properties"].get("location"),
        "catalogo": ds["properties"].get("location").split("Catalog=")[-1]
    }
    rows.append(row)

df = pd.DataFrame(rows)

# Opcional: mejor orden y nombres más ejecutivos
df = df.rename(columns={
    "nombre": "Datasource",
    "tipo": "Tipo",
    "driver": "Driver",
    "jdbc_location": "Cadena JDBC",
    "catalogo": "Archivo Mondrian"
})

df


,Datasource,Tipo,Driver,Cadena JDBC,Archivo Mondrian
0,declaracion101,OLAP,mondrian.olap4j.MondrianOlap4jDriver,jdbc:mondrian:Jdbc=jdbc:oracle:thin:/@INTER_BOLETIN_DIGITAL;JdbcDrivers=oracle.jdbc.driver.OracleDriver;Catalog=res:declaracion/Declaraciones101.mondrian.xml;,res:declaracion/Declaraciones101.mondrian.xml;
1,recaudacion,OLAP,mondrian.olap4j.MondrianOlap4jDriver,jdbc:mondrian:Jdbc=jdbc:oracle:thin:/@INTERSRI_BI_DWH;JdbcDrivers=oracle.jdbc.driver.OracleDriver;Catalog=res:recaudacion/Recaudacion_Boletin_Estrella_Investigador.xml;,res:recaudacion/Recaudacion_Boletin_Estrella_Investigador.xml;
2,declaracion104,OLAP,mondrian.olap4j.MondrianOlap4jDriver,jdbc:mondrian:Jdbc=jdbc:oracle:thin:/@INTER_BOLETIN_DIGITAL;JdbcDrivers=oracle.jdbc.driver.OracleDriver;Catalog=res:declaracion/Declaraciones104.mondrian.xml;,res:declaracion/Declaraciones104.mondrian.xml;
3,Contribuyentes,OLAP,mondrian.olap4j.MondrianOlap4jDriver,jdbc:mondrian:Jdbc=jdbc:oracle:thin:/@INTER_BOLETIN_DIGITAL;JdbcDrivers=oracle.jdbc.driver.OracleDriver;Catalog=res:declaracion/Contribuyentes.mondrian.xml;,res:declaracion/Contribuyentes.mondrian.xml;
4,declaracion103,OLAP,mondrian.olap4j.MondrianOlap4jDriver,jdbc:mondrian:Jdbc=jdbc:oracle:thin:/@INTER_BOLETIN_DIGITAL;JdbcDrivers=oracle.jdbc.driver.OracleDriver;Catalog=res:declaracion/Declaraciones103.mondrian.xml;,res:declaracion/Declaraciones103.mondrian.xml;


In [20]:
ds = "declaracion104"

r = session.get(
    f"https://srienlinea.sri.gob.ec/saiku/rest/saiku/anonymousUser/datasources/{ds}",
    headers=headers,
    verify=False
)

print(r.status_code)
print(r.json())


200
{'name': 'declaracion104', 'type': 'OLAP', 'properties': {'driver': 'mondrian.olap4j.MondrianOlap4jDriver', 'name': 'declaracion104', 'location': 'jdbc:mondrian:Jdbc=jdbc:oracle:thin:/@INTER_BOLETIN_DIGITAL;JdbcDrivers=oracle.jdbc.driver.OracleDriver;Catalog=res:declaracion/Declaraciones104.mondrian.xml;', 'type': 'OLAP'}}


C:\Users\dfreirev\AppData\Roaming\Python\Python311\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'srienlinea.sri.gob.ec'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


In [21]:
data = r.json()

# Aplanamos el JSON
df = pd.json_normalize(data)

# Opcional: renombrar columnas para que se vean más claras
df = df.rename(columns={
    "properties.driver": "driver",
    "properties.name": "datasource_name",
    "properties.location": "jdbc_location",
    "properties.type": "datasource_type"
})

df

,name,type,driver,datasource_name,jdbc_location,datasource_type
0,declaracion104,OLAP,mondrian.olap4j.MondrianOlap4jDriver,declaracion104,jdbc:mondrian:Jdbc=jdbc:oracle:thin:/@INTER_BOLETIN_DIGITAL;JdbcDrivers=oracle.jdbc.driver.OracleDriver;Catalog=res:declaracion/Declaraciones104.mondrian.xml;,OLAP


In [23]:
query_name = "q_d104"

payload = {
    "cube": "[D104]",
    "catalog": "Declaracion",
    "schema": "Declaracion",
    "connection": "declaracion104"
}

r = session.post(
    f"https://srienlinea.sri.gob.ec/saiku/rest/saiku/anonymousUser/query/{query_name}",
    json=payload,
    headers=headers,
    verify=False
)

print(r.status_code)
print(r.text)



500
<html><head><title>Error</title><script type="text/javascript" src="/ruxitagentjs_ICA15789NPQRTUVXfgqrux_10331260218130851.js" data-dtconfig="rid=RID_866026424|rpid=1463017999|domain=sri.gob.ec|reportUrl=/rb_bf24755trd|app=34a71c252bf77eda|owasp=1|featureHash=ICA15789NPQRTUVXfgqrux|msl=153600|srsr=30000|rdnt=1|uxrgce=1|cuc=4tn0mh4k|srms=2,1,0,0%2Ftextarea%2Cinput%2Cselect%2Coption;0%2Fdatalist;0%2Fform%20button;0%2F%5Bdata-dtrum-input%5D;0%2F.data-dtrum-input;1%2F%5Edata%28%28%5C-.%2B%24%29%7C%24%29|mel=100000|expw=1|dpvc=1|lastModification=1772467742187|postfix=4tn0mh4k|tp=500,50,0|srbbv=2|agentUri=/ruxitagentjs_ICA15789NPQRTUVXfgqrux_10331260218130851.js" data-config='{"revision":1772467742187,"beaconUri":"/rb_bf24755trd","agentUri":"/ruxitagentjs_ICA15789NPQRTUVXfgqrux_10331260218130851.js","environmentId":"4tn0mh4k","modules":"ICA15789NPQRTUVXfgqrux"}' data-envconfig='{"tracestateKeyPrefix":"933575cd-d80b97c5"}' data-appconfig='{"app":"34a71c252bf77eda"}'></script></head><body>

C:\Users\dfreirev\AppData\Roaming\Python\Python311\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'srienlinea.sri.gob.ec'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


HTTPError: 405 Client Error: Method Not Allowed for url: https://srienlinea.sri.gob.ec/saiku/rest/saiku/anonymousUser/query

In [93]:
pip install nest_asyncio --trusted-host pypi.org --trusted-host files.pythonhosted.org


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [108]:
import asyncio
import os
import pandas as pd
import nest_asyncio
from playwright.async_api import async_playwright

# 🔧 FIX CRÍTICO PARA WINDOWS
asyncio.set_event_loop_policy(asyncio.WindowsSelectorEventLoopPolicy())
nest_asyncio.apply()


In [110]:
asyncio.set_event_loop_policy(asyncio.WindowsSelectorEventLoopPolicy())
DOWNLOAD_DIR = "descargas_d104"
os.makedirs(DOWNLOAD_DIR, exist_ok=True)

SAIKU_URL = "https://srienlinea.sri.gob.ec/saiku"


In [115]:
def extraer_d104():
    with sync_playwright() as p:
        browser = p.chromium.launch(headless=True)
        context = browser.new_context(accept_downloads=True)
        page = context.new_page()

        page.goto(SAIKU_URL, timeout=60000)
        page.wait_for_load_state("networkidle")

        page.wait_for_selector("button[title='Export']", timeout=60000)

        with page.expect_download() as download_info:
            page.click("button[title='Export']")
            page.click("text=CSV")

        download = download_info.value
        csv_path = os.path.join(DOWNLOAD_DIR, download.suggested_filename)
        download.save_as(csv_path)

        browser.close()
        return csv_path

In [116]:
if __name__ == "__main__":
    csv = extraer_d104()
    df = pd.read_csv(csv)
    print(df.head())

Error: It looks like you are using Playwright Sync API inside the asyncio loop.
Please use the Async API instead.

In [88]:
df = pd.read_csv(csv_path)
df.head()

In [89]:
lista_claves_acceso = [
    "010120240109999999990012001001000000001123456781",
    "010120240109999999990012001001000000002123456782"
]

lista_xml_facturas = []

for clave in lista_claves_acceso:
    try:
        xml = consultar_xml_por_clave(clave)
        lista_xml_facturas.append(xml)
    except Exception as e:
        print(f"Error con clave {clave}: {e}")



Error con clave 010120240109999999990012001001000000001123456781: 404 Client Error: Not Found for url: https://srienlinea.sri.gob.ec/comprobantes-electronicos-ws/AutorizacionComprobantesOffline
Error con clave 010120240109999999990012001001000000002123456782: 404 Client Error: Not Found for url: https://srienlinea.sri.gob.ec/comprobantes-electronicos-ws/AutorizacionComprobantesOffline
